**Author**: Felipe Matheus
**Purpose**: Experiment launcher ("control panel") for the annealing_iacs surrogate.

This notebook does NOT contain pipeline logic. All the logic (Model A -> OOF ->
NNLS -> Model B -> calibration -> metrics -> persistence) lives in
`src/modeling/Experiments.py`, which orchestrates the existing `Modeling` and
`Evaluation` helpers. Here you only:

1. Load and prepare the data (once).
2. Define a base `ExperimentConfig`.
3. Define the grid of variations you want to sweep.
4. Run and inspect the central `experiments_log.csv`.

Results layout on disk:
```
models/annealing_iacs/experiments/
    experiments_log.csv          <- 1 row per run (the "results spreadsheet")
    <tag>__<hash>/               <- 1 folder per run
        config.yaml
        model_a/   model_b/
        artifacts.pkl
        leaderboard_autogluon.csv
```

# 1. Setup

In [ ]:
import logging
import os
import sys

import numpy as np
import pandas as pd

module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.processing.Processing import Processing
from src.feature_engineering.FeatureEngineering import FeatureEngineering
from src.modeling.Modeling import Modeling
from src.modeling.Evaluation import Evaluation
from src.modeling.Experiments import ExperimentConfig, ExperimentRunner

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)

%load_ext autoreload
%autoreload 2

proc = Processing()
feng = FeatureEngineering()
mdl = Modeling()
evl = Evaluation()
runner = ExperimentRunner(mdl, evl, models_root="../../models")

# 2. Data (same preparation as annealing_iacs.ipynb, run once)

In [ ]:
SCHEMA_DATE = "080626"
PATH_DATA_RAW = "../../data/raw"
FILE_NAME = "dataset_annealing_iacs.csv"
FILE_NAME_SCHEMA_DATA = f"schema_annealing_essays_{SCHEMA_DATE}.csv"

TARGET = "iacs_final"
ALL_FEATURES = ["purity", "iacs", "temperature", "time"]

# ---- Schema (essay) data: explicit is_essay marker ----
df_raw_schema = pd.read_csv(os.path.join(PATH_DATA_RAW, FILE_NAME_SCHEMA_DATA))
df_schema = df_raw_schema[ALL_FEATURES + [TARGET]].dropna()
df_schema["is_essay"] = True

# ---- Literature data ----
df_raw = pd.read_csv(os.path.join(PATH_DATA_RAW, FILE_NAME))
df_float = proc.df_to_float(df_raw, drop_cols=["DOI"], ignore_columns=["material"])
df_labeled = feng.label_element(df_float).drop_duplicates()
df_with_masks = feng.add_ratio_mask_column(
    feng.add_ratio_mask_column(df_labeled, "grain_size"), "iacs",
)
df_lit = df_with_masks[df_with_masks.has_Cu == True][ALL_FEATURES + [TARGET]]
df_lit["is_essay"] = False

assert df_lit[ALL_FEATURES + [TARGET]].isna().sum().sum() == 0, "NaNs in inputs"

# ---- Concat. weight_col is built INSIDE the runner from is_essay + config ----
df = pd.concat([df_schema, df_lit], ignore_index=True)
print(f"Dataset: {df.shape} | essays: {df.is_essay.sum()} | lit: {(~df.is_essay).sum()}")
df.head()

# 3. Base config

In [ ]:
base = ExperimentConfig(
    process="annealing_iacs",
    tag="annealing-v1",
    target=TARGET,
    features=tuple(ALL_FEATURES),
    # everything else uses the defaults; override here if needed, e.g.:
    # time_limit_a=120, weight_on_essay_rows=1.0, use_shared_folds=False,
)
print(base.run_id)

# 4. Single run (sanity check before any grid)

Always run the base config alone first. Then run it 2-3 more times with
`tag="annealing-v1-rep2"` etc. to measure run-to-run noise: AutoGluon under a
time budget is NOT deterministic, and at n~90 this noise is the floor below
which grid differences mean nothing.

In [ ]:
result = runner.run_experiment(df, base)
result["artifacts"]["metrics"]

# 5. Grid

Keys are `ExperimentConfig` field names; values are lists of variants.
`features` variants must be tuples. Already-completed runs are skipped
(`force=True` to redo).

In [ ]:
grid = {
    "time_limit_a": [60, 120, 300],
    "weight_on_essay_rows": [1.0, 3.0],
    "features": [
        ("purity", "iacs", "temperature", "time"),
        ("iacs", "temperature", "time"),
    ],
}
log = runner.run_grid(df, base, grid)   # 3 x 2 x 2 = 12 runs
log

# 6. Inspect results

In [ ]:
log = runner.load_log(base)

view_cols = [
    "run_id", "cfg_time_limit_a", "cfg_weight_on_essay_rows", "cfg_features",
    "rmse", "mae", "cov_0.9", "c_opt", "pct_truncated_aleat",
    "mean_sigma_epist", "mean_sigma_aleat", "elapsed_s",
]
log[[c for c in view_cols if c in log.columns]].sort_values("rmse")

In [ ]:
# Quick pivot: effect of one knob, marginalised over the others.
# Remember: compare against run-to-run noise (Section 4) before concluding.
log.groupby("cfg_time_limit_a")[["rmse", "mae", "cov_0.9"]].agg(["mean", "std"])

# 7. Load a winner for deployment / further analysis

Each run folder is self-contained: predictors + artifacts.pkl with weights,
`recalibration_c`, calibration tables.

In [ ]:
import pickle
from pathlib import Path
from autogluon.tabular import TabularPredictor

RUN_ID = log.sort_values("rmse").iloc[0]["run_id"]
run_dir = Path("../../models/annealing_iacs/experiments") / RUN_ID

with open(run_dir / "artifacts.pkl", "rb") as f:
    art = pickle.load(f)
predictor_a = TabularPredictor.load(str(run_dir / "model_a"))
predictor_b = TabularPredictor.load(str(run_dir / "model_b"))

print(RUN_ID)
art["calibration_after"]